In [1]:
import os
import json
from openai import OpenAI

# --- 1. 사용자로부터 받은 프롬프트 데이터를 변수에 저장 ---
# 따옴표 문제를 피하기 위해 삼중 따옴표를 사용합니다.
prompt_data_string = """
{
    "product_name": "덴마크 하이그릭요거트 400g",
    "persona_key": 1,
    "prompt": "# ROLE\\n당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 \\"제품 정보\\", 원시 \\"페르소나 데이터\\", 그리고 \\"시장 경쟁 환경\\"을 종합적으로 분석하여, 페르소나가 해당 제품의 잠재 구매자로서 어떤 특징을 보일지 예측하고, 그 결과를 하나의 완결된 JSON 객체로 생성하는 것입니다.\\n\\n# INSTRUCTION\\n아래의 모든 정보를 바탕으로, 페르소나가 해당 제품의 구매자로서 성립하는 **싱글턴 페르소나 JSON**을 생성하세요. 페르소나의 속성(attributes)과 제품의 특징(features)을 논리적으로 연결하여 구매 확률과 이유, 월별 구매 빈도를 예측해야 합니다.\\n\\n# INPUT DATA\\n## 1. 제품 정보\\n{\\n    \\"brand\\": \\"동원 F&B\\",\\n    \\"product_name\\": \\"덴마크 하이그릭요거트 400g\\",\\n    \\"category\\": \\"우유류 > 발효유 > 호상-중대용량\\",\\n    \\"features\\": [\\"건강식품\\", \\"고단백\\", \\"고소한맛\\", \\"높은 만족도\\"],\\n    \\"targeted_consumer\\": [\\"유당불내증\\"],\\n    \\"price_text\\": \\"3,980원\\",\\n    \\"advertise_info\\": \\"광고/프로모션: 2025년 6-7월, 일반인 광고\\"\\n}\\n\\n## 2. 페르소나 데이터\\n{\\n    \\"persona_key\\": 1,\\n    \\"attributes\\": {\\"gender\\": {\\"value\\": \\"남자\\", \\"weight\\": 0.0428...}, \\"age\\": {\\"value\\": \\"30대\\", \\"weight\\": 0.0428...}, \\"job\\": {\\"value\\": \\"사무 종사자\\", \\"weight\\": 0.0428...}, \\"education\\": {\\"value\\": \\"대학교 졸업(전문대졸/대학원생 포함)\\", \\"weight\\": 0.0428...}, \\"region\\": {\\"value\\": \\"경상북도\\", \\"weight\\": 0.0428...}, \\"household\\": {\\"value\\": \\"1세대가족\\", \\"weight\\": 0.0428...}, \\"marriage\\": {\\"value\\": \\"기혼\\", \\"weight\\": 0.0428...}, \\"brand_loyalty_scaled\\": {\\"value\\": 0.2089..., \\"weight\\": 0.0502...}, \\"cooking_convenience_scaled\\": {\\"value\\": 0.4295..., \\"weight\\": 0.1032...}, \\"health_orientation_scaled\\": {\\"value\\": 0.5343..., \\"weight\\": 0.1284...}, \\"hmr_preference_scaled\\": {\\"value\\": 0.6983..., \\"weight\\": 0.1678...}, \\"premium_orientation_scaled\\": {\\"value\\": 0.2866..., \\"weight\\": 0.0688...}, \\"price_sensitivity_scaled\\": {\\"value\\": 0.2290..., \\"weight\\": 0.0550...}, \\"variety_seeking_scaled\\": {\\"value\\": 0.5260..., \\"weight\\": 0.1264...}},\\n    \\"meta\\": {\\"cluster\\": 0, \\"label\\": \\"실속형 미식가\\", \\"Description\\": \\"편리성을 중시하면서도 새로운 맛과 제품을 시도하는 데 적극적인 소비자 그룹입니다. 가격에 민감하기보다는 효율성과 맛을 동시에 추구합니다.\\"}\\n}\\n\\n## 3. 시장 경쟁 환경\\n{\\n    \\"competitor_price_range_per_100g\\": \\"873원 ~ 1,422원\\"\\n}\\n\\n# OUTPUT FORMAT\\n반드시 아래와 같은 구조의 JSON 형식으로만 응답하세요. 다른 설명은 추가하지 마세요.\\n\\n{{\\n \\"product_name\\": \\"덴마크 하이그릭요거트 400g\\",\\n \\"persona_key\\": 1,\\n \\"purchase_behavior_prediction\\": {{\\n  \\"purchase_probability_pct\\": \\"<여기에 구매 확률(0-100)을 숫자로 예측>\\",\\n  \\"reason\\": \\"<여기에 페르소나 속성과 제품 특징, 시장 상황을 연결한 구매 결정 이유를 상세히 서술>\\",\\n  \\"monthly_purchase_frequency\\": {{\\n   \\"2024-07\\": 0, \\"2024-08\\": 0, \\"2024-09\\": 0, \\"2024-10\\": 0, \\"2024-11\\": 0, \\"2024-12\\": 0,\\n   \\"2025-01\\": 0, \\"2025-02\\": 0, \\"2025-03\\": 0, \\"2025-04\\": 0, \\"2025-05\\": 0, \\"2025-06\\": 0\\n  }}\\n }}\\n}}"
}
"""

# --- 2. API 요청 및 결과 출력 ---
try:
    print("OpenAI API에 요청을 보냅니다...")
    
    # 환경 변수에서 API 키를 불러와 클라이언트를 생성합니다.
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    # JSON 문자열을 파이썬 딕셔너리로 변환합니다.
    prompt_object = json.loads(prompt_data_string)
    
    # 실제 LLM에 보낼 프롬프트 텍스트를 추출합니다.
    prompt_text_to_send = prompt_object.get("prompt")

    # GPT-4o 모델에 채팅 형식으로 요청을 보냅니다.
    response = client.chat.completions.create(
        model="gpt-4o",  # GPT-4o 모델 사용 (JSON 모드 지원)
        messages=[
            {"role": "user", "content": prompt_text_to_send}
        ],
        response_format={"type": "json_object"},  # 응답을 JSON 형식으로 받도록 강제
        temperature=0.2  # 일관성 있는 답변을 위해 온도를 낮게 설정
    )

    # 응답 내용(JSON 문자열)을 추출합니다.
    llm_response_content = response.choices[0].message.content
    
    print("\n" + "="*50)
    print("✅ LLM으로부터 응답을 받았습니다!")
    print("="*50)
    
    # 받은 JSON 문자열을 파싱하여 가독성 좋게 출력합니다.
    parsed_response = json.loads(llm_response_content)
    print(json.dumps(parsed_response, ensure_ascii=False, indent=2))

except Exception as e:
    print(f"\n오류가 발생했습니다: {e}")

OpenAI API에 요청을 보냅니다...

✅ LLM으로부터 응답을 받았습니다!
{
  "product_name": "덴마크 하이그릭요거트 400g",
  "persona_key": 1,
  "purchase_behavior_prediction": {
    "purchase_probability_pct": 75,
    "reason": "페르소나는 '실속형 미식가'로서 편리성과 새로운 맛을 중시하며, 건강 지향적 성향이 강합니다. 덴마크 하이그릭요거트는 고단백 건강식품으로서 이러한 페르소나의 건강 지향적 성향과 잘 맞습니다. 또한, 가격 민감도가 낮고 효율성을 중시하는 페르소나에게 400g 대용량 제품은 적합합니다. 경쟁 제품의 가격 범위와 비교했을 때, 100g당 가격이 995원으로 중간 수준에 위치하여 가격 경쟁력도 갖추고 있습니다. 따라서, 이 제품은 페르소나의 구매 결정에 긍정적인 영향을 미칠 가능성이 높습니다.",
    "monthly_purchase_frequency": {
      "2024-07": 0,
      "2024-08": 0,
      "2024-09": 0,
      "2024-10": 0,
      "2024-11": 0,
      "2024-12": 0,
      "2025-01": 0,
      "2025-02": 0,
      "2025-03": 0,
      "2025-04": 0,
      "2025-05": 0,
      "2025-06": 1
    }
  }
}
